# Read Real Plans With EXPLAIN ANALYZE

Companion notebook for the [Reading EXPLAIN ANALYZE](https://codemia.io/courses/postgresql_deep_dive/pg_reading_explain_analyze) lesson on Codemia.

**What you will build**: an orders schema with a million rows on a live PostgreSQL server, and a small plan reader in Python that walks `EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)` output. It computes each node's self time across loops, ranks nodes by misestimate, and reports buffer hits and reads. You will use it on scans, joins and sorts, watch a sort spill to disk and fix it with `work_mem`, and repair a bad row estimate on correlated columns with extended statistics.

Runs on Colab with no GPU. Uses PostgreSQL, `psycopg2`, `numpy` and `matplotlib`.

## Setup

On Colab the next cell installs PostgreSQL with apt and starts it, which takes about a minute. Anywhere else, point the `PG_DSN` environment variable at a server you can create tables on, for example a Docker container started with `docker run -e POSTGRES_PASSWORD=postgres -p 5432:5432 postgres:14`.

In [ ]:
%pip install -q psycopg2-binary numpy matplotlib

In [ ]:
import os, sys, time, random, threading, subprocess
import psycopg2
import psycopg2.errors
import numpy as np
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    def sh(cmd):
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL)
    sh("apt-get -qq update && apt-get -qq install -y postgresql postgresql-contrib")
    sh("service postgresql start")
    sh("sudo -u postgres psql -qc \"ALTER USER postgres PASSWORD 'postgres'\"")

DSN = os.environ.get("PG_DSN", "host=localhost user=postgres password=postgres dbname=postgres")

def connect():
    conn = psycopg2.connect(DSN)
    conn.autocommit = True
    return conn

db = connect()

def q(sql, args=None, conn=None):
    with (conn or db).cursor() as cur:
        cur.execute(sql, args)
        return cur.fetchall() if cur.description else None

print(q("SELECT version()")[0][0])

## 1. An orders schema

Customers live in one of 50 cities, and each city belongs to exactly one country, so `city` and `country` are perfectly correlated. The planner assumes columns are independent unless told otherwise, which section 6 exploits. Orders have a skewed status column: almost all are `delivered`.

In [ ]:
q("DROP TABLE IF EXISTS orders, customers")
q("""CREATE TABLE customers (id int PRIMARY KEY, city int, country int, signup date)""")
q("""INSERT INTO customers
     SELECT g, g % 50, (g % 50) / 5, date '2020-01-01' + (g % 1500)
     FROM generate_series(1, 100000) g""")
q("""CREATE TABLE orders (id bigint PRIMARY KEY, customer_id int, status text,
                          amount numeric(10,2), created_at timestamptz)""")
q("""INSERT INTO orders
     SELECT g, 1 + (hashint4(g::int) & 2147483647) % 100000,
            CASE WHEN g % 100 = 0 THEN 'pending' WHEN g % 100 < 3 THEN 'cancelled' ELSE 'delivered' END,
            (g % 50000) / 100.0,
            timestamptz '2024-01-01' + (g || ' seconds')::interval * 30
     FROM generate_series(1, 1000000) g""")
q("VACUUM ANALYZE customers"); q("VACUUM ANALYZE orders")
print(q("SELECT relname, reltuples::bigint, pg_size_pretty(pg_relation_size(oid)) FROM pg_class WHERE relname IN ('orders','customers')"))

## 2. A plan reader

`EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)` returns the same tree as the text form, with every number as a field. Two rules matter when reading it:

- `Actual Total Time` is per loop, and includes the children. A node's own work is its total times its loops, minus the same quantity for each child.
- `Actual Rows` is also per loop. Compare `Plan Rows` against `Actual Rows`, both per loop, to judge an estimate.

The reader turns off parallel query for each statement, so every plan runs in one process and loops mean only what the join made them mean. Parallel plans add a `Gather` node and count each worker as a loop, which is worth learning separately.

In [ ]:
import json

def explain(sql, settings=None):
    conn = connect()
    with conn.cursor() as cur:
        cur.execute("SET max_parallel_workers_per_gather = 0")
        for k, v in (settings or {}).items():
            cur.execute(f"SET {k} = %s", (v,))
        cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) " + sql)
        plan = cur.fetchone()[0][0]
        cur.execute("EXPLAIN (ANALYZE, BUFFERS, COSTS OFF, TIMING OFF) " + sql)
        text = "\n".join(r[0] for r in cur.fetchall())
    conn.close()
    return plan, text

def walk(node, depth=0, out=None):
    out = [] if out is None else out
    loops = node.get("Actual Loops", 1)
    total = node["Actual Total Time"] * loops
    kids = node.get("Plans", [])
    child = sum(k["Actual Total Time"] * k.get("Actual Loops", 1) for k in kids)
    est, act = node["Plan Rows"], node["Actual Rows"]
    out.append(dict(depth=depth, type=node["Node Type"], rel=node.get("Relation Name", node.get("Index Name", "")),
                    self_ms=max(total - child, 0.0), loops=loops, est=est, act=act,
                    misestimate=max(est, 1) / max(act, 1) if act or est else 1.0,
                    hit=node.get("Shared Hit Blocks", 0), read=node.get("Shared Read Blocks", 0),
                    temp=node.get("Temp Written Blocks", 0)))
    for k in kids:
        walk(k, depth + 1, out)
    return out

def report(sql, settings=None, show_text=False):
    plan, text = explain(sql, settings)
    nodes = walk(plan["Plan"])
    if show_text:
        print(text, "\n")
    print("%-34s %9s %6s %9s %9s %8s %7s %6s" % ("node", "self ms", "loops", "est rows", "act rows", "est/act", "hit", "temp"))
    for n in nodes:
        name = "  " * n["depth"] + n["type"] + (" " + n["rel"] if n["rel"] else "")
        print("%-34s %9.2f %6d %9d %9d %8.2f %7d %6d" % (name[:34], n["self_ms"], n["loops"], n["est"], n["act"], n["misestimate"], n["hit"], n["temp"]))
    print("execution %.1f ms" % plan["Execution Time"])
    return plan, nodes

## 3. Scans: sequential, index, bitmap

The same predicate shape gets three different plans depending on how many rows match. A rare status is fetched through an index; a common one reads the whole table, because random index lookups for most of the rows would cost more than one sequential pass.

In [ ]:
q("CREATE INDEX IF NOT EXISTS orders_status ON orders (status)")
q("CREATE INDEX IF NOT EXISTS orders_customer ON orders (customer_id)")
q("ANALYZE orders")
for status in ("pending", "delivered"):
    print(f"--- status = '{status}'")
    report(f"SELECT count(*), sum(amount) FROM orders WHERE status = '{status}'")
    print()
print("--- one customer")
report("SELECT * FROM orders WHERE customer_id = 4242", show_text=True);

The status estimates are close because the column has a most common values list in `pg_stats`. The single customer lookup is a bitmap scan: the index finds the handful of matching row locations, the bitmap sorts them by page, and the heap pages are read in order. The `Buffers` line in the text plan counts 8 kB pages, and `shared hit` means they were already in shared buffers.

## 4. Joins and self time across loops

A nested loop runs its inner side once per outer row. The inner node's `Actual Total Time` is per loop, so the reader multiplies by loops before subtracting from the parent, otherwise the slowest node looks cheap.

In [ ]:
sql = """SELECT c.city, count(*), sum(o.amount)
         FROM customers c JOIN orders o ON o.customer_id = c.id
         WHERE c.signup = date '2021-06-01'
         GROUP BY c.city"""
plan, nodes = report(sql, show_text=True)
worst = max(nodes, key=lambda n: n["self_ms"])
print("\nslowest node by self time:", worst["type"], worst["rel"], "%.2f ms over %d loops" % (worst["self_ms"], worst["loops"]))

Now force a hash join and a merge join for the same query and compare. The planner picked one of these on cost; the measured times show whether it was right on this data.

In [ ]:
variants = {
    "planner choice": {},
    "hash join": {"enable_nestloop": "off", "enable_mergejoin": "off"},
    "merge join": {"enable_nestloop": "off", "enable_hashjoin": "off"},
    "nested loop": {"enable_hashjoin": "off", "enable_mergejoin": "off"},
}
times = {}
for label, s in variants.items():
    best = min(explain(sql, s)[0]["Execution Time"] for _ in range(3))
    times[label] = best
    print("%-15s %8.2f ms" % (label, best))

plt.figure(figsize=(6.5, 3))
plt.barh(list(times), list(times.values()), color=["black", "tab:blue", "tab:orange", "tab:green"])
plt.xlabel("execution time, ms (best of 3)"); plt.title("Same query, three join methods"); plt.tight_layout(); plt.show()

## 5. A sort that spills to disk

When the rows to sort do not fit in `work_mem`, the sort writes runs to temporary files and merges them. The text plan says `Sort Method: external merge  Disk: ...` and the JSON reports temp blocks written. Sweep `work_mem` and watch the method and the time change.

In [ ]:
sort_sql = "SELECT id, amount FROM orders ORDER BY amount DESC, id LIMIT 1000000"
rows = []
for wm in ("1MB", "4MB", "16MB", "32MB", "64MB", "128MB"):
    plan, _ = explain(sort_sql, {"work_mem": wm})
    def find(n):
        return n if n["Node Type"] == "Sort" else next((f for k in n.get("Plans", []) if (f := find(k))), None)
    sort = find(plan["Plan"])
    rows.append((wm, sort["Sort Method"], sort.get("Sort Space Type"), sort.get("Sort Space Used"), plan["Execution Time"]))
    print("work_mem %-6s %-16s %-7s %7s kB   %7.1f ms" % rows[-1])

plt.figure(figsize=(6.5, 3))
plt.plot([r[0] for r in rows], [r[4] for r in rows], marker="o")
for r in rows:
    plt.annotate(r[2], (r[0], r[4]), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
plt.ylabel("execution ms"); plt.xlabel("work_mem"); plt.title("Sorting a million rows: disk until it fits in memory"); plt.tight_layout(); plt.show()

`work_mem` is per sort or hash node per query, not per server, so raising it globally multiplies across connections. Raise it for the session that runs the heavy report with `SET work_mem`, or for a role, rather than in `postgresql.conf`.

With a `LIMIT` that is small, the same sort becomes a `top-N heapsort` that keeps only N rows in memory, and the spill disappears regardless of `work_mem`:

In [ ]:
report("SELECT id, amount FROM orders ORDER BY amount DESC, id LIMIT 20", {"work_mem": "1MB"}, show_text=True);

## 6. A misestimate from correlated columns, and the fix

City determines country, so `city = 7 AND country = 1` matches exactly the customers in city 7. The planner multiplies the two selectivities as if they were independent and expects about a tenth of that. Rank nodes by how far off the estimate is.

In [ ]:
corr_sql = """SELECT count(*) FROM customers c JOIN orders o ON o.customer_id = c.id
              WHERE c.city = 7 AND c.country = 1"""
plan, nodes = report(corr_sql)
bad = max(nodes, key=lambda n: max(n["misestimate"], 1 / n["misestimate"]))
print("\nworst estimate: %s %s, estimated %d, actual %d" % (bad["type"], bad["rel"], bad["est"], bad["act"]))

An underestimate on the outer side of a join is the dangerous kind: it can make the planner choose a nested loop that then runs many more times than planned. `CREATE STATISTICS` with `dependencies` tells the planner that one column determines the other.

In [ ]:
q("CREATE STATISTICS IF NOT EXISTS cust_city_country (dependencies) ON city, country FROM customers")
q("ANALYZE customers")
print(q("SELECT dependencies FROM pg_stats_ext WHERE statistics_name = 'cust_city_country'")[0][0])
plan, nodes = report(corr_sql)
scan = next(n for n in nodes if n["rel"] == "customers")
print("\ncustomers scan now: estimated %d, actual %d" % (scan["est"], scan["act"]))

The dependency degree of 1.0 from `city` to `country` means the second predicate adds no selectivity, and the estimate now matches.

## 7. Exercises

1. Add `CREATE INDEX ON orders (customer_id) INCLUDE (amount)`, run `VACUUM orders`, and rerun section 4. Which node changes to an index only scan, and what happens to `Heap Fetches` in the text plan?
2. Create a copy with `CREATE TABLE orders_copy AS SELECT * FROM orders`, then scan it twice with `report`. The first scan shows `read` blocks that the second shows as `hit`. On Colab, run `!service postgresql restart` between scans to see a fully cold read.
3. Extend `walk` to report hash nodes: `Hash Batches` and `Peak Memory Usage`. Find the `work_mem` at which the hash join in section 4 needs a single batch when you force it.